# Inference Latency Benchmarks

Baseline measurements before KV cache implementation. Run this after any inference optimization to quantify improvement.

In [ ]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path("../../../../").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import torch
import numpy as np

from midi_gen.data_management.testing import get_seed_tokens
from midi_gen.model.models.GPTMidiV1 import GPTMidiV1
from midi_gen.model.inference.base_inference import create_sample_tokens
from midi_gen.data_management.tokenizing import create_vocabulary

MODEL_PATH   = PROJECT_ROOT / "src/midi_gen/model/models/midiv1_best_1.pt"
DATASET_PATH = PROJECT_ROOT / "data/tokenized_dataset.npy"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

checkpoint = torch.load(MODEL_PATH, map_location=device)
state_dict = checkpoint["model_state_dict"]
if any(k.startswith("module.") for k in state_dict):
    state_dict = {k.removeprefix("module."): v for k, v in state_dict.items()}
max_seq_len = state_dict["transformer_blocks.0.rope_cos"].shape[2]
model = GPTMidiV1(max_seq_len=max_seq_len)
model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()
print(f"Model loaded on {device} (max_seq_len={max_seq_len})")

## Benchmark

Generates `N_SAMPLES` samples, each up to `MAX_LENGTH` tokens, seeded from different rows in the dataset.
Records wall-clock time per sample and breaks it down per token.

In [2]:
N_SAMPLES  = 5
MAX_LENGTH = 1024   # keep short for baseline — increase to 1024/2048 post-cache
TEMPERATURE = 0.9
TOP_P       = 0.9

results = []

for i in range(N_SAMPLES):

    # sync before timing so MPS/CUDA ops are complete
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

    t_start = time.perf_counter()
    tokens = create_sample_tokens(model, max_length=MAX_LENGTH, temperature=TEMPERATURE, top_p=TOP_P)
    
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()
    t_end = time.perf_counter()

    elapsed      = t_end - t_start
    n_generated  = tokens.shape[1]
    ms_per_token = (elapsed / n_generated) * 1000 if n_generated > 0 else 0

    results.append({
        "sample":        i + 1,
        "generated":     n_generated,
        "total_tokens":  tokens.shape[1],
        "elapsed_s":     elapsed,
        "ms_per_token":  ms_per_token,
    })

    print(f"Sample {i+1}: {n_generated} tokens generated in {elapsed:.2f}s  ({ms_per_token:.1f} ms/token)")
    torch.mps.empty_cache()

Sample 1: 117 tokens generated in 1.26s  (10.7 ms/token)
Sample 2: 1024 tokens generated in 5.43s  (5.3 ms/token)
Sample 3: 1024 tokens generated in 5.35s  (5.2 ms/token)
Sample 4: 1024 tokens generated in 5.34s  (5.2 ms/token)
Sample 5: 1024 tokens generated in 5.36s  (5.2 ms/token)


## Summary

In [ ]:
avg_elapsed      = np.mean([r["elapsed_s"]    for r in results])
avg_ms_per_token = np.mean([r["ms_per_token"] for r in results])
avg_generated    = np.mean([r["generated"]    for r in results])

print("=" * 50)
print(f"  Samples:              {N_SAMPLES}")
print(f"  Max length:           {MAX_LENGTH} tokens")
print(f"  Avg tokens generated: {avg_generated:.0f}")
print(f"  Avg total time:       {avg_elapsed:.2f}s")
print(f"  Avg time per token:   {avg_ms_per_token:.1f} ms")
print(f"  Avg tokens per sec:   {1000 / avg_ms_per_token:.1f}")
print("=" * 50)
print()
print("Per-sample breakdown:")
print(f"  {'Sample':<8} {'Generated':<12} {'Time (s)':<12} {'ms/token'}")
print(f"  {'-'*8} {'-'*12} {'-'*12} {'-'*8}")
for r in results:
    print(f"  {r['sample']:<8} {r['generated']:<12} {r['elapsed_s']:<12.2f} {r['ms_per_token']:.1f}")